# Нарезка данных на train/val/test с правильной стратификацией

Разбиение исходных данных (training.csv) на три группы (60/20/20) с учётом:
- Каждый клиент попадает только в одну группу
- Fraud rate совпадает на уровне клиентов
- Fraud rate совпадает на уровне транзакций

## 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 2. Загрузка и исследование данных

In [2]:
# Загружаем исходные данные
df = pd.read_csv('full.csv')

print(f"Размер датасета: {df.shape}")

print(f"\nКолонки: {df.columns.tolist()}")
print(f"\nПервые несколько строк:")
print(df.head())
print(f"\nТипы данных:")
print(df.dtypes)
print(f"\nПропущенные значения:")
print(df.isnull().sum())

Размер датасета: (95662, 16)

Колонки: ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']

Первые несколько строк:
         TransactionId         BatchId  ... PricingStrategy FraudResult
0  TransactionId_76871   BatchId_36123  ...               2           0
1  TransactionId_73770   BatchId_15642  ...               2           0
2  TransactionId_26203   BatchId_53941  ...               2           0
3    TransactionId_380  BatchId_102363  ...               2           0
4  TransactionId_28195   BatchId_38780  ...               2           0

[5 rows x 16 columns]

Типы данных:
TransactionId               str
BatchId                     str
AccountId                   str
SubscriptionId              str
CustomerId                  str
CurrencyCode                str
CountryCode               int64
Pr

## 3. Очистка данных

In [3]:
# НЕ удаляем колонки - работаем с исходными данными
print(f"Используем все колонки: {df.columns.tolist()}")

Используем все колонки: ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']


## 4. Анализ для стратификации

In [4]:
# Анализируем данные по клиентам и мошенничеству
print("=" * 60)
print("ИСХОДНЫЕ СТАТИСТИКИ")
print("=" * 60)

print(f"\nОбщие данные:")
print(f"Всего транзакций: {len(df)}")
print(f"Уникальных клиентов: {df['CustomerId'].nunique()}")
print(f"Fraud rate (транзакции): {df['FraudResult'].mean():.4f}")

# Группируем по клиентам для анализа
customer_stats = df.groupby('CustomerId').agg({
    'FraudResult': ['sum', 'count', 'mean']  # sum=кол-во фрод, count=всего транзакций, mean=fraud rate
}).reset_index()

customer_stats.columns = ['CustomerId', 'fraud_count', 'total_transactions', 'fraud_rate']

print(f"\nПо клиентам:")
print(f"Клиентов с мошенничеством: {(customer_stats['fraud_count'] > 0).sum()}")
print(f"Средний fraud rate по клиентам: {customer_stats['fraud_rate'].mean():.4f}")
print(f"Распределение fraud_rate:")
print(customer_stats['fraud_rate'].describe())

print(f"\nРаспределение числа транзакций на клиента:")
print(customer_stats['total_transactions'].describe())

ИСХОДНЫЕ СТАТИСТИКИ

Общие данные:
Всего транзакций: 95662
Уникальных клиентов: 3742
Fraud rate (транзакции): 0.0020

По клиентам:
Клиентов с мошенничеством: 54
Средний fraud rate по клиентам: 0.0056
Распределение fraud_rate:
count    3742.000000
mean        0.005556
std         0.064539
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: fraud_rate, dtype: float64

Распределение числа транзакций на клиента:
count    3742.000000
mean       25.564404
std        96.929602
min         1.000000
25%         2.000000
50%         7.000000
75%        20.000000
max      4091.000000
Name: total_transactions, dtype: float64


## 5. Балансированное разбиение клиентов на 60/20/20

`qcut` по `fraud_count` и `fraud_rate` для всех клиентов здесь не подходит: почти все клиенты имеют `fraud_count = 0`, поэтому квантили схлопываются в одну страту. Разделяем задачу на две части: отдельно балансируем редких fraud-клиентов, а non-fraud клиентов режем по бинам размера клиента (`total_transactions`).

In [5]:
"""
Балансировка клиентов для сильно разреженного fraud таргета.

Идея:
1. Fraud-клиентов всего 54, поэтому не дробим их qcut-ом. Фиксируем целевое
   число fraud-клиентов в train/val/test и ищем распределение, которое
   балансирует сумму fraud_count, fraud_rate и число транзакций.
2. Non-fraud клиентов много, поэтому режем их на 5 бинов по total_transactions.
   В каждом таком бине сохраняем пропорцию 60/20/20.
3. Все разбиение идет по CustomerId: один клиент попадает только в один split.
"""

SPLIT_NAMES = ["train", "val", "test"]
SPLIT_RATIOS = np.array([0.6, 0.2, 0.2])
RANDOM_STATE = 42
N_FRAUD_ASSIGNMENT_TRIALS = 200_000


def target_split_counts(total):
    """Return exact train/val/test counts with train=60%, val/test split evenly."""
    train = int(total * 0.6)
    remaining = total - train
    val = remaining // 2
    test = total - train - val
    return pd.Series([train, val, test], index=SPLIT_NAMES, dtype=int)


def proportional_bin_counts(size, column_targets):
    """Allocate one bin across splits proportionally, then fix rounding."""
    raw = size * SPLIT_RATIOS
    counts = np.floor(raw).astype(int)
    missing = size - counts.sum()
    for split_idx in np.argsort(-(raw - counts))[:missing]:
        counts[split_idx] += 1
    return pd.Series(counts, index=column_targets.index, dtype=int)


def fix_bin_target_totals(bin_targets, column_targets):
    """Adjust rounded per-bin targets so split columns sum to exact targets."""
    bin_targets = bin_targets.copy()

    while not bin_targets.sum(axis=0).equals(column_targets):
        diff = column_targets - bin_targets.sum(axis=0)
        to_split = diff[diff > 0].index[0]
        from_split = diff[diff < 0].index[0]

        candidate_bins = bin_targets.index[bin_targets[from_split] > 0]
        move_bin = bin_targets.loc[candidate_bins, from_split].idxmax()
        bin_targets.loc[move_bin, from_split] -= 1
        bin_targets.loc[move_bin, to_split] += 1

    return bin_targets


target_customer_counts = target_split_counts(len(customer_stats))
target_fraud_customer_counts = target_split_counts(int((customer_stats["fraud_count"] > 0).sum()))
target_transaction_counts = target_split_counts(int(customer_stats["total_transactions"].sum()))
target_fraud_transaction_counts = target_split_counts(int(customer_stats["fraud_count"].sum()))

fraud_customers = customer_stats[customer_stats["fraud_count"] > 0].reset_index(drop=True)
non_fraud_customers = customer_stats[customer_stats["fraud_count"] == 0].copy()

non_fraud_customers["tx_size_bin"] = pd.qcut(
    non_fraud_customers["total_transactions"],
    q=5,
    labels=False,
    duplicates="drop"
).astype(int)

fraud_customers["fraud_count_bucket"] = pd.cut(
    fraud_customers["fraud_count"],
    bins=[0, 1, 2, 7, np.inf],
    labels=["1", "2", "3-7", "8+"],
    right=True
)

print("Целевые размеры split:")
print(f"Клиенты:           {target_customer_counts.to_dict()}")
print(f"Fraud-клиенты:     {target_fraud_customer_counts.to_dict()}")
print(f"Транзакции:        {target_transaction_counts.to_dict()}")
print(f"Fraud-транзакции:  {target_fraud_transaction_counts.to_dict()}")

print("\nИсходные группы клиентов:")
print(f"Non-fraud клиентов: {len(non_fraud_customers)}")
print(f"Fraud клиентов:     {len(fraud_customers)}")

print("\nNon-fraud клиенты по tx_size_bin:")
print(non_fraud_customers["tx_size_bin"].value_counts().sort_index())

print("\nFraud-клиенты по fraud_count_bucket:")
print(fraud_customers["fraud_count_bucket"].value_counts().sort_index())

# Fraud-клиенты: фиксируем 32/11/11 клиентов и ищем лучший баланс по fraud_count.
base_assignment = np.concatenate([
    np.full(target_fraud_customer_counts[split_name], split_idx)
    for split_idx, split_name in enumerate(SPLIT_NAMES)
])

target_fraud_tx = target_fraud_transaction_counts.to_numpy(dtype=float)
target_fraud_rate_sum = fraud_customers["fraud_rate"].sum() * (
    target_fraud_customer_counts / len(fraud_customers)
).to_numpy(dtype=float)
target_fraud_customer_tx = fraud_customers["total_transactions"].sum() * (
    target_fraud_customer_counts / len(fraud_customers)
).to_numpy(dtype=float)

rng = np.random.default_rng(RANDOM_STATE)
best_score = np.inf
best_assignment = None

for _ in range(N_FRAUD_ASSIGNMENT_TRIALS):
    assignment = rng.permutation(base_assignment)
    fraud_tx_by_split = np.bincount(
        assignment,
        weights=fraud_customers["fraud_count"],
        minlength=len(SPLIT_NAMES)
    )
    fraud_rate_sum_by_split = np.bincount(
        assignment,
        weights=fraud_customers["fraud_rate"],
        minlength=len(SPLIT_NAMES)
    )
    fraud_customer_tx_by_split = np.bincount(
        assignment,
        weights=fraud_customers["total_transactions"],
        minlength=len(SPLIT_NAMES)
    )

    fraud_tx_error = ((fraud_tx_by_split - target_fraud_tx) / target_fraud_tx) ** 2
    fraud_rate_error = ((fraud_rate_sum_by_split - target_fraud_rate_sum) / target_fraud_rate_sum) ** 2
    fraud_customer_tx_error = ((fraud_customer_tx_by_split - target_fraud_customer_tx) / target_fraud_customer_tx) ** 2

    score = fraud_tx_error.sum() * 10 + fraud_rate_error.sum() * 2 + fraud_customer_tx_error.sum() * 0.3
    if score < best_score:
        best_score = score
        best_assignment = assignment.copy()

split_states = {
    split_name: {"ids": [], "customers": 0, "transactions": 0}
    for split_name in SPLIT_NAMES
}

for split_idx, split_name in enumerate(SPLIT_NAMES):
    split_fraud_customers = fraud_customers.loc[best_assignment == split_idx]
    split_states[split_name]["ids"].extend(split_fraud_customers["CustomerId"].tolist())
    split_states[split_name]["customers"] += len(split_fraud_customers)
    split_states[split_name]["transactions"] += int(split_fraud_customers["total_transactions"].sum())

# Non-fraud клиенты: целевые count по каждому tx_size_bin, затем раскладка крупных
# клиентов в split с наибольшим дефицитом транзакций.
remaining_customer_targets = target_customer_counts - pd.Series({
    split_name: split_states[split_name]["customers"]
    for split_name in SPLIT_NAMES
})

non_fraud_bin_targets = pd.DataFrame(
    [
        proportional_bin_counts(len(bin_customers), remaining_customer_targets)
        for _, bin_customers in non_fraud_customers.groupby("tx_size_bin")
    ],
    index=sorted(non_fraud_customers["tx_size_bin"].unique())
)
non_fraud_bin_targets.index.name = "tx_size_bin"
non_fraud_bin_targets = fix_bin_target_totals(non_fraud_bin_targets, remaining_customer_targets)

for tx_size_bin, bin_customers in non_fraud_customers.groupby("tx_size_bin"):
    quotas = non_fraud_bin_targets.loc[tx_size_bin].copy()
    bin_customers = bin_customers.sort_values("total_transactions", ascending=False)

    for row in bin_customers.itertuples(index=False):
        candidates = [split_name for split_name in SPLIT_NAMES if quotas[split_name] > 0]
        selected_split = max(
            candidates,
            key=lambda split_name: (
                (target_transaction_counts[split_name] - split_states[split_name]["transactions"])
                / target_transaction_counts[split_name]
            )
        )
        quotas[selected_split] -= 1
        split_states[selected_split]["ids"].append(row.CustomerId)
        split_states[selected_split]["customers"] += 1
        split_states[selected_split]["transactions"] += int(row.total_transactions)

train_ids = set(split_states["train"]["ids"])
val_ids = set(split_states["val"]["ids"])
test_ids = set(split_states["test"]["ids"])

train_customers = customer_stats[customer_stats["CustomerId"].isin(train_ids)].copy()
val_customers = customer_stats[customer_stats["CustomerId"].isin(val_ids)].copy()
test_customers = customer_stats[customer_stats["CustomerId"].isin(test_ids)].copy()

customer_split = pd.concat([
    pd.DataFrame({"CustomerId": sorted(train_ids), "split": "train"}),
    pd.DataFrame({"CustomerId": sorted(val_ids), "split": "val"}),
    pd.DataFrame({"CustomerId": sorted(test_ids), "split": "test"}),
], ignore_index=True)

customer_stats_with_split = customer_stats.merge(customer_split, on="CustomerId", how="left")

print(f"\n{'='*60}")
print("РАЗМЕР SPLIT (по клиентам):")
print(f"{'='*60}")
for split_name, split_customers in [
    ("train", train_customers),
    ("val", val_customers),
    ("test", test_customers),
]:
    print(
        f"{split_name:5s}: {len(split_customers):4d} "
        f"({len(split_customers)/len(customer_stats)*100:4.1f}%)"
    )

overlap_train_val = len(train_ids & val_ids)
overlap_train_test = len(train_ids & test_ids)
overlap_val_test = len(val_ids & test_ids)
covered_customers = len(train_ids | val_ids | test_ids)

print("\nПроверка CustomerId:")
print(f"Train ∩ Val:  {overlap_train_val}")
print(f"Train ∩ Test: {overlap_train_test}")
print(f"Val ∩ Test:   {overlap_val_test}")
print(f"Покрыто клиентов: {covered_customers} из {len(customer_stats)}")

assert overlap_train_val == 0
assert overlap_train_test == 0
assert overlap_val_test == 0
assert covered_customers == len(customer_stats)

print(f"\n{'='*60}")
print("БАЛАНС ПО КЛИЕНТАМ:")
print(f"{'='*60}")
for split_name, split_customers in [
    ("train", train_customers),
    ("val", val_customers),
    ("test", test_customers),
]:
    fraud_transactions = int(split_customers["fraud_count"].sum())
    total_transactions = int(split_customers["total_transactions"].sum())
    fraud_customer_count = int((split_customers["fraud_count"] > 0).sum())
    print(
        f"{split_name:5s}: "
        f"fraud-клиентов={fraud_customer_count:2d}, "
        f"fraud-транзакций={fraud_transactions:3d}, "
        f"транзакций={total_transactions:5d}, "
        f"tx fraud rate={fraud_transactions/total_transactions:.6f}, "
        f"customer fraud rate={split_customers['fraud_rate'].mean():.6f}"
    )

print(f"\nВсего fraud-транзакций: {int(customer_stats['fraud_count'].sum())}")
print(f"Score распределения fraud-клиентов: {best_score:.6f}")


Целевые размеры split:
Клиенты:           {'train': 2245, 'val': 748, 'test': 749}
Fraud-клиенты:     {'train': 32, 'val': 11, 'test': 11}
Транзакции:        {'train': 57397, 'val': 19132, 'test': 19133}
Fraud-транзакции:  {'train': 115, 'val': 39, 'test': 39}

Исходные группы клиентов:
Non-fraud клиентов: 3688
Fraud клиентов:     54

Non-fraud клиенты по tx_size_bin:
tx_size_bin
0    948
1    659
2    655
3    703
4    723
Name: count, dtype: int64

Fraud-клиенты по fraud_count_bucket:
fraud_count_bucket
1      28
2      10
3-7    11
8+      5
Name: count, dtype: int64

РАЗМЕР SPLIT (по клиентам):
train: 2245 (60.0%)
val  :  748 (20.0%)
test :  749 (20.0%)

Проверка CustomerId:
Train ∩ Val:  0
Train ∩ Test: 0
Val ∩ Test:   0
Покрыто клиентов: 3742 из 3742

БАЛАНС ПО КЛИЕНТАМ:
train: fraud-клиентов=32, fraud-транзакций=114, транзакций=57364, tx fraud rate=0.001987, customer fraud rate=0.005350
val  : fraud-клиентов=11, fraud-транзакций= 40, транзакций=19162, tx fraud rate=0.002087, cus

## 5.1 Проверка бинов после разбиения

In [6]:
# Подробная проверка бинов после split
print("=" * 60)
print("ПРОВЕРКА БИНОВ ПОСЛЕ РАЗБИЕНИЯ")
print("=" * 60)

non_fraud_with_split = non_fraud_customers.merge(customer_split, on="CustomerId", how="left")
non_fraud_bin_actual = pd.crosstab(
    non_fraud_with_split["tx_size_bin"],
    non_fraud_with_split["split"]
).reindex(columns=SPLIT_NAMES)

print("\nNon-fraud клиенты: целевые count по tx_size_bin:")
print(non_fraud_bin_targets[SPLIT_NAMES])

print("\nNon-fraud клиенты: фактические count по tx_size_bin:")
print(non_fraud_bin_actual[SPLIT_NAMES])

print("\nОтклонение fact - target:")
print(non_fraud_bin_actual[SPLIT_NAMES] - non_fraud_bin_targets[SPLIT_NAMES])

fraud_with_split = fraud_customers.merge(customer_split, on="CustomerId", how="left")
fraud_bucket_actual = pd.crosstab(
    fraud_with_split["fraud_count_bucket"],
    fraud_with_split["split"]
).reindex(columns=SPLIT_NAMES)

print("\nFraud-клиенты по fraud_count_bucket:")
print(fraud_bucket_actual[SPLIT_NAMES])

split_metrics = []
for split_name, split_customers in [
    ("train", train_customers),
    ("val", val_customers),
    ("test", test_customers),
]:
    split_metrics.append({
        "split": split_name,
        "customers": len(split_customers),
        "target_customers": int(target_customer_counts[split_name]),
        "transactions": int(split_customers["total_transactions"].sum()),
        "target_transactions": int(target_transaction_counts[split_name]),
        "fraud_customers": int((split_customers["fraud_count"] > 0).sum()),
        "target_fraud_customers": int(target_fraud_customer_counts[split_name]),
        "fraud_transactions": int(split_customers["fraud_count"].sum()),
        "target_fraud_transactions": int(target_fraud_transaction_counts[split_name]),
    })

split_metrics = pd.DataFrame(split_metrics).set_index("split")
split_metrics["transaction_delta"] = split_metrics["transactions"] - split_metrics["target_transactions"]
split_metrics["fraud_transaction_delta"] = split_metrics["fraud_transactions"] - split_metrics["target_fraud_transactions"]

print("\nИтоговые метрики split:")
print(split_metrics)

assert (non_fraud_bin_actual[SPLIT_NAMES] == non_fraud_bin_targets[SPLIT_NAMES]).all().all()
assert (split_metrics["customers"] == split_metrics["target_customers"]).all()
assert (split_metrics["fraud_customers"] == split_metrics["target_fraud_customers"]).all()


ПРОВЕРКА БИНОВ ПОСЛЕ РАЗБИЕНИЯ

Non-fraud клиенты: целевые count по tx_size_bin:
             train  val  test
tx_size_bin                  
0              569  188   191
1              395  132   132
2              393  131   131
3              422  141   140
4              434  145   144

Non-fraud клиенты: фактические count по tx_size_bin:
split        train  val  test
tx_size_bin                  
0              569  188   191
1              395  132   132
2              393  131   131
3              422  141   140
4              434  145   144

Отклонение fact - target:
split        train  val  test
tx_size_bin                  
0                0    0     0
1                0    0     0
2                0    0     0
3                0    0     0
4                0    0     0

Fraud-клиенты по fraud_count_bucket:
split               train  val  test
fraud_count_bucket                  
1                      19    4     5
2                       4    4     2
3-7                   

## 6. Распределение транзакций по расщепам

In [7]:
# Распределяем транзакции по расщепам на основе CustomerId
train_df = df[df['CustomerId'].isin(train_ids)].copy()
val_df = df[df['CustomerId'].isin(val_ids)].copy()
test_df = df[df['CustomerId'].isin(test_ids)].copy()

print(f"\n{'='*60}")
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print(f"{'='*60}")

print(f"\nРазмер по транзакциям:")
print(f"Train: {len(train_df):6d} ({len(train_df)/len(df)*100:5.1f}%)")
print(f"Val:   {len(val_df):6d} ({len(val_df)/len(df)*100:5.1f}%)")
print(f"Test:  {len(test_df):6d} ({len(test_df)/len(df)*100:5.1f}%)")
print(f"Всего: {len(df):6d}")

print(f"\nFraud rate (по транзакциям):")
print(f"Train: {train_df['FraudResult'].mean():.4f} ({train_df['FraudResult'].sum():.0f} фрод)")
print(f"Val:   {val_df['FraudResult'].mean():.4f} ({val_df['FraudResult'].sum():.0f} фрод)")
print(f"Test:  {test_df['FraudResult'].mean():.4f} ({test_df['FraudResult'].sum():.0f} фрод)")
print(f"Исход: {df['FraudResult'].mean():.4f} ({df['FraudResult'].sum():.0f} фрод)")

print(f"\nFraud rate (по клиентам):")
train_fraud_rate_by_customer = train_df.groupby('CustomerId')['FraudResult'].mean().mean()
val_fraud_rate_by_customer = val_df.groupby('CustomerId')['FraudResult'].mean().mean()
test_fraud_rate_by_customer = test_df.groupby('CustomerId')['FraudResult'].mean().mean()

print(f"Train: {train_fraud_rate_by_customer:.4f}")
print(f"Val:   {val_fraud_rate_by_customer:.4f}")
print(f"Test:  {test_fraud_rate_by_customer:.4f}")
print(f"Исход: {df.groupby('CustomerId')['FraudResult'].mean().mean():.4f}")

print(f"\nУникальные клиенты:")
print(f"Train: {train_df['CustomerId'].nunique()}")
print(f"Val:   {val_df['CustomerId'].nunique()}")
print(f"Test:  {test_df['CustomerId'].nunique()}")
print(f"Всего: {df['CustomerId'].nunique()}")


ФИНАЛЬНАЯ СТАТИСТИКА

Размер по транзакциям:
Train:  57364 ( 60.0%)
Val:    19162 ( 20.0%)
Test:   19136 ( 20.0%)
Всего:  95662

Fraud rate (по транзакциям):
Train: 0.0020 (114 фрод)
Val:   0.0021 (40 фрод)
Test:  0.0020 (39 фрод)
Исход: 0.0020 (193 фрод)

Fraud rate (по клиентам):
Train: 0.0053
Val:   0.0057
Test:  0.0060
Исход: 0.0056

Уникальные клиенты:
Train: 2245
Val:   748
Test:  749
Всего: 3742


## 7. Сохранение файлов

In [8]:
# Сохраняем RAW файлы (со всеми колонками)
train_df.to_csv('train_raw.csv', index=False)
val_df.to_csv('val_raw.csv', index=False)
test_df.to_csv('test_raw.csv', index=False)

print(f"\n{'='*60}")
print("✓ RAW ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!")
print(f"{'='*60}")
print(f"train_raw.csv - {len(train_df)} строк, {train_df.shape[1]} колонок")
print(f"val_raw.csv   - {len(val_df)} строк, {val_df.shape[1]} колонок")
print(f"test_raw.csv  - {len(test_df)} строк, {test_df.shape[1]} колонок")


✓ RAW ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!
train_raw.csv - 57364 строк, 16 колонок
val_raw.csv   - 19162 строк, 16 колонок
test_raw.csv  - 19136 строк, 16 колонок


## 8. Удаление ненужных колонок и сохранение финальных файлов

In [9]:
# Удаляем ненужные колонки из RAW файлов
cols_to_drop = ['CountQuantile', 'BatchId', 'SubscriptionId', 'AccountId', 'CurrencyCode', 'CountryCode', 'Value']

# Удаляем только существующие колонки
cols_to_drop = [col for col in cols_to_drop if col in train_df.columns]

print(f"Удаляем колонки: {cols_to_drop}")

# Создаём очищенные версии
train_clean = train_df.drop(columns=cols_to_drop).copy()
val_clean = val_df.drop(columns=cols_to_drop).copy()
test_clean = test_df.drop(columns=cols_to_drop).copy()

print(f"Осталось колонок: {train_clean.shape[1]}")
print(f"Новые колонки: {train_clean.columns.tolist()}")

# Сохраняем очищенные файлы (без суффикса raw)
train_clean.to_csv('train.csv', index=False)
val_clean.to_csv('val.csv', index=False)
test_clean.to_csv('test.csv', index=False)

print(f"\n{'='*60}")
print("✓ ОЧИЩЕННЫЕ ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!")
print(f"{'='*60}")
print(f"train.csv - {len(train_clean)} строк, {train_clean.shape[1]} колонок")
print(f"val.csv   - {len(val_clean)} строк, {val_clean.shape[1]} колонок")
print(f"test.csv  - {len(test_clean)} строк, {test_clean.shape[1]} колонок")

print(f"\n{'='*60}")
print("ИТОГОВАЯ СТАТИСТИКА ОЧИЩЕННЫХ ФАЙЛОВ")
print(f"{'='*60}")

print(f"\nFraud rate (по транзакциям):")
print(f"Train: {train_clean['FraudResult'].mean():.4f}")
print(f"Val:   {val_clean['FraudResult'].mean():.4f}")
print(f"Test:  {test_clean['FraudResult'].mean():.4f}")

Удаляем колонки: ['BatchId', 'SubscriptionId', 'AccountId', 'CurrencyCode', 'CountryCode', 'Value']
Осталось колонок: 10
Новые колонки: ['TransactionId', 'CustomerId', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']

✓ ОЧИЩЕННЫЕ ФАЙЛЫ УСПЕШНО СОХРАНЕНЫ!
train.csv - 57364 строк, 10 колонок
val.csv   - 19162 строк, 10 колонок
test.csv  - 19136 строк, 10 колонок

ИТОГОВАЯ СТАТИСТИКА ОЧИЩЕННЫХ ФАЙЛОВ

Fraud rate (по транзакциям):
Train: 0.0020
Val:   0.0021
Test:  0.0020
